# Clase 5: máquinas de soporte vectorial y kernels

Visualizaciones interactivas para acompañar las notas de la clase 5. Las figuras siguen el orden y los colores de las notas (clase $y = -1$: círculos azules, clase $y = 1$: triángulos rojos; los vectores de soporte llevan un círculo negro alrededor) y usan los mismos datos: se importan desde `notes/scripts/05-svm-figuras.py`, el script que genera las figuras del PDF. Los ejemplos que no están en las notas se indican en su sección.

En las figuras con fondo de color, el color indica el score $\mathbf{w}^T\mathbf{x} + b$: rojo donde es positivo (se predice $y = 1$), azul donde es negativo y blanco cerca de la frontera. La línea negra es la frontera ($\text{score} = 0$) y las punteadas son $\text{score} = \pm 1$.

Para usarlo en clase basta con ejecutar todas las celdas al inicio y luego mover los sliders.

In [17]:
%matplotlib inline
import importlib.util
import timeit
import warnings
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, widgets
from sklearn.exceptions import ConvergenceWarning
from sklearn.svm import SVC

# Datos y modelos de las notas (los mismos que generan las figuras del PDF)
ruta = Path("../notes/scripts/05-svm-figuras.py")
spec = importlib.util.spec_from_file_location("fig05", ruta)
fig05 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fig05)

C_NEG, C_POS, VERDE, NARANJA = "#1f77b4", "#d62728", "tab:green", "tab:orange"
C_DURO = fig05.C_DURO  # C muy grande: margen duro
NORMA_W = r"$\|\mathbf{w}\|$"

plt.rcParams.update({
    "figure.figsize": (11, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})
# Con datos que no son separables, el margen duro (C enorme) puede no converger
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def puntos(ax, X, y, tam=110):
    """Clase -1: círculos azules. Clase 1: triángulos rojos. Grandes, para el proyector."""
    ax.scatter(X[y == -1, 0], X[y == -1, 1], s=tam, marker="o", color=C_NEG, ec="white", lw=1.2, zorder=3, label="$y = -1$")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=tam * 1.15, marker="^", color=C_POS, ec="white", lw=1.2, zorder=3, label="$y = 1$")


def soporte(ax, sv, tam=110, color="black"):
    """Círculo vacío alrededor de los vectores de soporte, como en las notas."""
    sv = np.asarray(sv).reshape(-1, 2)
    ax.scatter(sv[:, 0], sv[:, 1], s=tam * 3, facecolor="none", ec=color, lw=1.8, zorder=4)


def recta(ax, w, b, c=0.0, **estilo):
    """Dibuja la recta w^T x + b = c (también si es vertical)."""
    w = np.asarray(w, dtype=float)
    p = (c - b) * w / (w @ w)  # punto de la recta más cercano al origen
    d = np.array([-w[1], w[0]]) / np.linalg.norm(w)
    t = np.array([-100.0, 100.0])
    ax.plot(p[0] + t * d[0], p[1] + t * d[1], **estilo)


def calle(ax, w, b, xlim=(0, 5), ylim=(0, 5)):
    """Frontera w^T x + b = 0 (negra) y rectas w^T x + b = ±1 (punteadas)."""
    for c in (-1, 1):
        recta(ax, w, b, c, color="0.35", ls="--", lw=1.5)
    recta(ax, w, b, color="black", lw=2.5)
    ax.set(xlim=xlim, ylim=ylim)


def fondo(ax, score, xlim, ylim, n=250, lineas=True):
    """Colorea el plano según score(X): rojo si es positivo, azul si es negativo.

    El color se satura en |score| = 2. Con lineas=True dibuja también la
    frontera (score = 0) y las curvas score = ±1."""
    g1, g2 = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    Z = score(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)
    ax.imshow(Z, extent=(*xlim, *ylim), origin="lower", cmap="RdBu_r", vmin=-2, vmax=2, alpha=0.3, aspect="auto")
    if lineas:
        ax.contour(g1, g2, Z, levels=[-1, 0, 1], colors=["0.35", "black", "0.35"],
                   linestyles=["--", "-", "--"], linewidths=[1.5, 2.5, 1.5])
    ax.set(xlim=xlim, ylim=ylim)


def svm_lineal(X, y, C=C_DURO):
    """SVM lineal ajustada igual que en el script de las figuras (tol pequeña).

    Con datos que no son separables y C enorme el solver tarda mucho, así que
    se limita el número de iteraciones."""
    return SVC(kernel="linear", C=C, tol=1e-8, max_iter=200_000).fit(X, y)


def porcentaje(x):
    """Redondea como Typst (0.925 -> 93%), para que coincida con las notas."""
    return f"{int(np.floor(100 * x + 0.5))}%"


def alphas(modelo, m):
    """Los α_i de los m ejemplos (cero para los que no son vectores de soporte)."""
    a = np.zeros(m)
    a[modelo.support_] = np.abs(modelo.dual_coef_.ravel())  # dual_coef_ = α_i y_i
    return a

## 1. Margen funcional y margen geométrico (Figuras 1 y 2)

Los datos separables de las notas. Escribimos los parámetros como $\mathbf{w} = c\,(\cos\varphi, \sin\varphi)$ y $b = c\,b_0$: el ángulo fija la dirección de $\mathbf{w}$, $b_0$ desplaza la frontera, y $c > 0$ multiplica $\mathbf{w}$ y $b$ a la vez. Con $c = 1$ se tiene $\|\mathbf{w}\| = 1$.

A la izquierda, la frontera, las rectas $\mathbf{w}^T\mathbf{x} + b = \pm 1$ y, en verde, la distancia del peor ejemplo a la frontera (el segmento AB de la Figura 2). Los ejemplos mal clasificados aparecen con un anillo negro. A la derecha, el margen funcional $\hat\gamma^{(i)} = y^{(i)}(\mathbf{w}^T\mathbf{x}^{(i)} + b)$ de cada ejemplo, de menor a mayor; el peor ejemplo está a la izquierda, con borde verde.

Qué observar:

- $\hat\gamma^{(i)} > 0$ exactamente para los ejemplos bien clasificados. Al girar o desplazar la frontera hasta que un ejemplo quede del lado equivocado, su barra se vuelve negativa.
- Con $c = 1$, los dos márgenes coinciden: $\hat\gamma = \gamma$.
- Al cambiar $c$ la frontera no se mueve, pero todas las barras se estiran o se encogen: el margen funcional depende de la escala. El margen geométrico $\gamma = \hat\gamma / \|\mathbf{w}\|$ (el segmento verde) no cambia.
- Las rectas punteadas sí se mueven con $c$. Para cierto valor de $c$ pasan justo por el peor ejemplo: es la escala con $\hat\gamma = 1$, el truco de la Parte 2.

In [18]:
X_sep, y_sep = fig05.X_sep, fig05.y_sep


def ver_margenes(angulo=45, b0=-3.3, c=1.0):
    u = np.array([np.cos(np.deg2rad(angulo)), np.sin(np.deg2rad(angulo))])
    w, b = c * u, c * b0
    funcional = y_sep * (X_sep @ w + b)        # γ̂^(i)
    geometrico = funcional / np.linalg.norm(w)  # γ^(i)
    peor = np.argmin(funcional)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    fondo(ax1, lambda P: P @ w + b, (0, 5), (0, 5), lineas=False)
    calle(ax1, w, b)
    puntos(ax1, X_sep, y_sep)
    ax1.scatter(*X_sep[funcional < 0].T, s=320, facecolor="none", ec="black", lw=2, zorder=4)
    # Segmento AB: del peor ejemplo a su proyección sobre la frontera
    A = X_sep[peor]
    B = A - (A @ w + b) * w / (w @ w)
    ax1.plot(*np.column_stack([A, B]), color=VERDE, lw=3.5, zorder=5)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$",
            title=rf"$\hat\gamma$ = {funcional.min():.2f},   " + NORMA_W + rf" = {np.linalg.norm(w):.2f},   $\gamma$ = {geometrico.min():.2f}")

    orden = np.argsort(funcional)
    colores = np.where(y_sep[orden] == 1, C_POS, C_NEG)
    barras = ax2.bar(np.arange(len(orden)), funcional[orden], color=colores, alpha=0.8)
    barras[0].set(edgecolor=VERDE, linewidth=3)
    ax2.axhline(0, color="black", lw=1)
    ax2.axhline(1, color="0.35", ls="--", lw=1.5, label=r"$\hat\gamma^{(i)} = 1$")
    ax2.set(ylim=(-4, 13), xticks=[], xlabel="Ejemplos, de menor a mayor margen",
            ylabel=r"$\hat\gamma^{(i)} = y^{(i)}(\mathbf{w}^T\mathbf{x}^{(i)} + b)$", title="Margen funcional de cada ejemplo")
    ax2.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

In [19]:
interact(ver_margenes,
         angulo=widgets.IntSlider(min=0, max=90, value=45, description="Ángulo de w"),
         b0=widgets.FloatSlider(min=-4.5, max=-2, step=0.05, value=-3.3, description="b₀"),
         c=widgets.FloatLogSlider(value=1, base=10, min=-0.6, max=0.6, step=0.02, description="c"));

interactive(children=(IntSlider(value=45, description='Ángulo de w', max=90), FloatSlider(value=-3.3, descript…

## 2. El clasificador de margen óptimo (Figura 3)

Fijemos solo la dirección de $\mathbf{w}$, con el slider del ángulo. Entre todas las fronteras con esa dirección, la de mayor margen es la que queda justo a la mitad entre el positivo más cercano y el negativo más cercano. A la izquierda se dibuja esa frontera, escalada para que su margen funcional sea 1: las rectas punteadas pasan por los ejemplos más cercanos (marcados con un círculo) y están a distancia $\gamma = 1/\|\mathbf{w}\|$ de la frontera. A la derecha, el mejor margen $\gamma$ que se puede obtener con cada dirección.

Qué observar:

- Cada dirección tiene su propio mejor margen. Hay direcciones con las que ninguna recta separa los datos ($\gamma < 0$ en la curva).
- El clasificador de margen óptimo es la dirección donde la curva alcanza su máximo. La línea punteada vertical marca la solución que encuentra `SVC`, y coincide con el máximo de la curva.
- En el máximo, la calle es lo más ancha posible y la tocan exactamente los tres vectores de soporte de la Figura 3. Fuera del máximo, la tocan solo dos ejemplos.

In [20]:
opt = fig05.separables["opt"]
angulo_opt = np.degrees(np.arctan2(opt["w"][1], opt["w"][0]))


def mejor_calle(angulo):
    """Para w en la dirección del ángulo: la dirección u, el mejor b (con ||u|| = 1) y su margen."""
    u = np.array([np.cos(np.deg2rad(angulo)), np.sin(np.deg2rad(angulo))])
    p = X_sep @ u
    pos_min, neg_max = p[y_sep == 1].min(), p[y_sep == -1].max()
    return u, -(pos_min + neg_max) / 2, (pos_min - neg_max) / 2


angulos = np.linspace(-30, 120, 601)
curva_gamma = np.array([mejor_calle(a)[2] for a in angulos])


def ver_margen_optimo(angulo=15.0):
    if abs(angulo - angulo_opt) < 0.3:  # el slider no cae justo en el óptimo
        angulo = angulo_opt
    u, b, gamma = mejor_calle(angulo)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    if gamma > 0:
        # Escala con margen funcional 1: las rectas ±1 quedan a distancia γ de la frontera
        w, b = u / gamma, b / gamma
        fondo(ax1, lambda P: P @ w + b, (0, 5), (0, 5), lineas=False)
        calle(ax1, w, b)
        soporte(ax1, X_sep[np.isclose(y_sep * (X_sep @ w + b), 1, atol=1e-6)])
        ax1.set_title(rf"$\gamma$ = {gamma:.3f},   " + NORMA_W + rf" = 1/$\gamma$ = {1 / gamma:.2f}")
    else:
        recta(ax1, u, b, color="black", lw=2.5)
        ax1.set(xlim=(0, 5), ylim=(0, 5), title="Con esta dirección ninguna recta separa los datos")
    puntos(ax1, X_sep, y_sep)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$")

    ax2.plot(angulos, curva_gamma, color=C_NEG, lw=2.5)
    ax2.axhline(0, color="0.7", lw=1)
    ax2.axvline(angulo_opt, color="0.5", ls=":", lw=2, label=f"SVC: ángulo = {angulo_opt:.1f}°, γ = {opt['margen']:.3f}")
    ax2.plot(angulo, gamma, "o", color=C_NEG, ms=11)
    ax2.set(xlim=(angulos[0], angulos[-1]), ylim=(-1, 1.1), xlabel="Ángulo de w (grados)", ylabel=r"Mejor margen $\gamma$",
            title="Mejor margen posible con cada dirección")
    ax2.legend(loc="lower center")
    plt.tight_layout()
    plt.show()

In [21]:
interact(ver_margen_optimo, angulo=widgets.FloatSlider(min=-30, max=120, step=0.5, value=15, description="Ángulo de w"));

interactive(children=(FloatSlider(value=15.0, description='Ángulo de w', max=120.0, min=-30.0, step=0.5), Outp…

## 3. El problema dual

### Por qué $\mathbf{w}$ es una combinación de los ejemplos (Figura 4)

Los ejemplos de la Figura 4, todos sobre una recta que pasa por el origen (gris). Empezamos con el $\mathbf{w}_\parallel$ óptimo, en la dirección de la recta, y le sumamos una componente $t\,\mathbf{u}_\perp$ perpendicular a ella: $\mathbf{w} = \mathbf{w}_\parallel + t\,\mathbf{u}_\perp$, con el mismo $b$.

Qué observar:

- La frontera (naranja) gira, pero los scores de los seis ejemplos (abajo a la izquierda) no cambian: la componente perpendicular no afecta ninguna restricción.
- $\|\mathbf{w}\|$ sí crece, así que el margen geométrico $\gamma = 1/\|\mathbf{w}\|$ baja. A la derecha, $\gamma$ es máximo con $t = 0$: en el óptimo, $\mathbf{w}$ está en la dirección de los ejemplos.

In [22]:
# Los mismos valores que la Figura 4 de las notas
u_span, u_perp = np.array([0.894, 0.447]), np.array([-0.447, 0.894])
t_span = np.array([-2.4, -1.8, -1.3, 1.0, 1.6, 2.3])
X_span = t_span[:, None] * u_span
y_span = np.array([-1, -1, -1, 1, 1, 1])
w_par, b_span = 0.8696 * u_span, 0.1304
ts = np.linspace(-2, 2, 201)
gamma_ts = np.array([np.min(y_span * (X_span @ (w_par + t * u_perp) + b_span)) / np.linalg.norm(w_par + t * u_perp) for t in ts])


def ver_perpendicular(t=0.6):
    w = w_par + t * u_perp
    scores = X_span @ w + b_span
    gamma = np.min(y_span * scores) / np.linalg.norm(w)
    lims = dict(xlim=(-2.5, 2.5), ylim=(-2, 2))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw=dict(width_ratios=[1.25, 1]))
    ax1.plot([-3, 3], [-1.5, 1.5], color="0.6", ls=":", lw=1.5)
    recta(ax1, w_par, b_span, color="black", lw=2, alpha=0.35)
    for c in (-1, 1):
        recta(ax1, w, b_span, c, color=NARANJA, ls="--", lw=1.3)
    recta(ax1, w, b_span, color=NARANJA, lw=2.5)
    puntos(ax1, X_span, y_span, tam=140)
    p0 = -0.15 * u_span
    ax1.annotate("", xy=p0 + 1.4 * w, xytext=p0, arrowprops=dict(arrowstyle="-|>", lw=2.5, color=NARANJA))
    ax1.annotate("", xy=p0 + 1.4 * w_par, xytext=p0, arrowprops=dict(arrowstyle="-|>", lw=2, color="0.3"))
    ax1.text(*(p0 + 1.5 * w_par + [0.05, -0.25]), r"$\mathbf{w}_\parallel$", fontsize=14, color="0.3")
    ax1.text(*(p0 + 1.5 * w + [0.05, 0.05]), r"$\mathbf{w}$", fontsize=14, color=NARANJA)
    ax1.text(-2.4, -1.9, "scores: " + "  ".join(f"{s:+.2f}" for s in scores), fontsize=11, family="monospace")
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$", title=f"t = {t:.2f}:  " + NORMA_W + f" = {np.linalg.norm(w):.2f},  γ = {gamma:.3f}", **lims)

    ax2.plot(ts, gamma_ts, color=NARANJA, lw=2.5)
    ax2.plot(t, gamma, "o", color=NARANJA, ms=11)
    ax2.set(xlabel=r"Componente perpendicular $t$", ylabel=r"Margen geométrico $\gamma$", ylim=(0, 1.2),
            title=r"$\gamma$ es máximo con $t = 0$")
    plt.tight_layout()
    plt.show()

In [23]:
interact(ver_perpendicular, t=widgets.FloatSlider(min=-2, max=2, step=0.05, value=0.6, description="t"));

interactive(children=(FloatSlider(value=0.6, description='t', max=2.0, min=-2.0, step=0.05), Output()), _dom_c…

### Vectores de soporte (Figura 3)

Volvemos a los datos separables y al clasificador de margen óptimo. Escogemos un ejemplo en el menú y lo movemos con los sliders; cada vez se vuelve a entrenar la SVM. La frontera original aparece en gris claro. A la derecha, los $\alpha_i$ del problema dual, uno por ejemplo.

Qué observar:

- Solo los vectores de soporte tienen $\alpha_i > 0$, y se cumple $\sum_i \alpha_i y^{(i)} = 0$. El $\mathbf{w}$ que se reconstruye con $\sum_i \alpha_i y^{(i)}\mathbf{x}^{(i)}$ es el mismo que da `SVC`.
- Si movemos un ejemplo que no es vector de soporte, la frontera no cambia, siempre que no entre en la calle: su $\alpha_i$ es 0 y no aparece en $\mathbf{w}$.
- Si movemos un vector de soporte, la frontera se mueve con él. Si lo alejamos de la frontera, otro ejemplo puede tomar su lugar como vector de soporte.
- Si lo llevamos al otro lado, los datos dejan de ser separables y el margen duro ya no tiene solución (Parte 5).

In [24]:
m_sep = len(y_sep)
modelo_sep = svm_lineal(X_sep, y_sep)
es_sv = np.isin(np.arange(m_sep), modelo_sep.support_)
opciones_sv = [(f"Ejemplo {i}: vector de soporte", i) for i in np.flatnonzero(es_sv)]
opciones_sv += [(f"Ejemplo {i}", i) for i in np.flatnonzero(~es_sv)]


def ver_soporte(i=int(modelo_sep.support_[0]), dx=0.0, dy=0.0):
    X = X_sep.copy()
    X[i] += (dx, dy)
    modelo = svm_lineal(X, y_sep)
    w, b = modelo.coef_.ravel(), modelo.intercept_[0]
    separable = np.all(y_sep * (X @ w + b) >= 1 - 1e-3)
    a = alphas(modelo, m_sep)
    w_dual = (a * y_sep) @ X

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw=dict(width_ratios=[1, 1.15]))
    recta(ax1, modelo_sep.coef_.ravel(), modelo_sep.intercept_[0], color="0.75", lw=2)
    if separable:
        fondo(ax1, lambda P: P @ w + b, (0, 5), (0, 5), lineas=False)
        calle(ax1, w, b)
        soporte(ax1, modelo.support_vectors_)
        ax1.set_title(f"{len(modelo.support_)} vectores de soporte,  γ = {1 / np.linalg.norm(w):.3f}")
    else:
        ax1.set(xlim=(0, 5), ylim=(0, 5), title="Ya no es separable: el margen duro no tiene solución")
    puntos(ax1, X, y_sep)
    if dx or dy:
        ax1.annotate("", xy=X[i], xytext=X_sep[i], arrowprops=dict(arrowstyle="-|>", lw=2, color=VERDE), zorder=5)
    ax1.scatter(*X[i], s=500, facecolor="none", ec=VERDE, lw=2.5, zorder=5)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$")

    colores = np.where(y_sep == 1, C_POS, C_NEG)
    ax2.bar(np.arange(m_sep), a, color=colores)
    ax2.bar(i, a[i], color=colores[i], ec=VERDE, lw=3)
    ax2.set(xlabel="Ejemplo $i$", ylabel=r"$\alpha_i$", xticks=range(0, m_sep, 3),
            title=rf"$\sum_i \alpha_i y^{{(i)}}$ = {np.sum(a * y_sep):.4f}")
    ax2.text(0.02, 0.97, "w de SVC      = ({:6.3f}, {:6.3f})\n".format(*w)
             + "Σ α_i y_i x_i = ({:6.3f}, {:6.3f})".format(*w_dual),
             transform=ax2.transAxes, va="top", family="monospace", fontsize=11)
    if separable:
        ax2.set_ylim(0, max(1.4 * a.max(), 0.1))
    plt.tight_layout()
    plt.show()

In [25]:
interact(ver_soporte,
         i=widgets.Dropdown(options=opciones_sv, value=int(modelo_sep.support_[0]), description="Ejemplo"),
         dx=widgets.FloatSlider(min=-2, max=2, step=0.05, value=0, description="Mover en x₁"),
         dy=widgets.FloatSlider(min=-2, max=2, step=0.05, value=0, description="Mover en x₂"));

interactive(children=(Dropdown(description='Ejemplo', options=(('Ejemplo 4: vector de soporte', np.int64(4)), …

## 4. Kernels

### El kernel polinomial sin calcular $\phi$

Para $\phi(\mathbf{x})$ con todos los productos $x_i x_j$, las notas muestran que $\phi(\mathbf{x})^T\phi(\mathbf{z}) = (\mathbf{x}^T\mathbf{z})^2$. Comprobémoslo con vectores al azar, y comparemos el tiempo de las dos formas de calcularlo: construir $\phi$ cuesta del orden de $d^2$ operaciones, el kernel del orden de $d$.

In [26]:
rng = np.random.default_rng(0)
print(f"{'d':>5}  {'componentes de ϕ':>16}  {'ϕ(x)ᵀϕ(z)':>12}  {'(xᵀz)²':>12}  {'tiempo con ϕ':>13}  {'tiempo con K':>13}")
for d in (3, 10, 100, 1000, 3000):
    x, z = rng.normal(size=(2, d))
    con_phi = lambda: np.outer(x, x).ravel() @ np.outer(z, z).ravel()  # ϕ(x) = (x_i x_j)_{i,j}
    con_k = lambda: (x @ z) ** 2
    t_phi = timeit.timeit(con_phi, number=20) / 20
    t_k = timeit.timeit(con_k, number=20) / 20
    print(f"{d:>5}  {d * d:>16,}  {con_phi():>12.4f}  {con_k():>12.4f}  {t_phi * 1e6:>10.1f} µs  {t_k * 1e6:>10.1f} µs")

    d  componentes de ϕ     ϕ(x)ᵀϕ(z)        (xᵀz)²   tiempo con ϕ   tiempo con K
    3                 9        0.0996        0.0996         7.9 µs         1.4 µs
   10               100        4.8681        4.8681         5.5 µs         1.1 µs
  100            10,000        9.4532        9.4532       138.2 µs         1.4 µs
 1000         1,000,000     3611.5864     3611.5864     19190.6 µs         2.7 µs
 3000         9,000,000       13.6362       13.6362     49102.1 µs         3.5 µs


### Llevar los datos a más dimensiones (Figura 5)

Un ejemplo que no está en las notas (los mismos datos circulares del notebook de la clase 4). La clase $y = 1$ está dentro de un círculo y ninguna recta la separa de la clase $y = -1$. Con $d = 2$, el kernel $K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^T\mathbf{z})^2$ corresponde a

$$
\phi(\mathbf{x}) = (x_1^2,\ \sqrt{2}\,x_1 x_2,\ x_2^2),
$$

que tiene tres componentes y se puede dibujar. La SVM se entrena solo con el kernel (`SVC(kernel="poly", degree=2, gamma=1, coef0=0)`), y después reconstruimos $\mathbf{w} = \sum_i \alpha_i y^{(i)} \phi(\mathbf{x}^{(i)})$ para dibujar la frontera en el espacio de $\phi$.

Qué observar al girar la vista de la derecha:

- En el espacio de $\phi$, la frontera es un plano, y separa las dos clases: los puntos de la clase $y = 1$ quedan abajo y los de la clase $y = -1$ arriba.
- Ese mismo plano, visto en el espacio original (izquierda), es una elipse.
- Con elevación 0 y azimut −90 se mira el plano de canto, en los ejes $(x_1^2, x_2^2)$: se ve como una recta que separa las clases.
- Las curvas punteadas de la izquierda ($\text{score} = \pm 1$) también son elipses: son los planos paralelos $\mathbf{w}^T\phi(\mathbf{x}) + b = \pm 1$.

In [27]:
rng_circ = np.random.default_rng(4)


def en_polar(radio, angulo):
    return np.column_stack([radio * np.cos(angulo), radio * np.sin(angulo)])


# Clase 1: 14 puntos en un disco de radio 1.2 (patrón de girasol). Clase -1: 18 en un anillo.
n_dentro, n_fuera = 14, 18
k = np.arange(n_dentro)
dentro = en_polar(1.2 * np.sqrt((k + 0.5) / n_dentro), 2.39996 * k + rng_circ.uniform(-0.15, 0.15, n_dentro))
angulos_fuera = np.linspace(0, 2 * np.pi, n_fuera, endpoint=False) + rng_circ.uniform(-0.12, 0.12, n_fuera)
fuera = en_polar(rng_circ.uniform(2.1, 2.7, n_fuera), angulos_fuera)
X_circ = np.vstack([fuera, dentro])
y_circ = np.concatenate([-np.ones(n_fuera), np.ones(n_dentro)])


def phi_cuad(X):
    """ϕ(x) = (x_1², √2 x_1 x_2, x_2²), que cumple ϕ(x)ᵀϕ(z) = (xᵀz)²."""
    return np.column_stack([X[:, 0] ** 2, np.sqrt(2) * X[:, 0] * X[:, 1], X[:, 1] ** 2])


modelo_circ = SVC(kernel="poly", degree=2, gamma=1, coef0=0, C=C_DURO).fit(X_circ, y_circ)
w_phi = modelo_circ.dual_coef_.ravel() @ phi_cuad(modelo_circ.support_vectors_)  # Σ α_i y_i ϕ(x_i)
b_phi = modelo_circ.intercept_[0]
diferencia = np.abs(phi_cuad(X_circ) @ w_phi + b_phi - modelo_circ.decision_function(X_circ)).max()
print(f"w = {np.round(w_phi, 3)},  b = {b_phi:.3f}")
print(f"Diferencia máxima entre w^T ϕ(x) + b y el score de SVC: {diferencia:.1e}")


def ver_levantar(elevacion=20, azimut=-60):
    fig = plt.figure(figsize=(13, 5.6))
    ax1 = fig.add_subplot(1, 2, 1)
    fondo(ax1, modelo_circ.decision_function, (-3.2, 3.2), (-3.2, 3.2))
    puntos(ax1, X_circ, y_circ, tam=120)
    soporte(ax1, modelo_circ.support_vectors_, tam=120)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$", title="Espacio original: la frontera es una elipse")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    F = phi_cuad(X_circ)
    g1, g2 = np.meshgrid(np.linspace(0, 7.5, 30), np.linspace(-5.5, 5.5, 30))
    g3 = -(b_phi + w_phi[0] * g1 + w_phi[1] * g2) / w_phi[2]  # el plano w^T ϕ + b = 0
    g3[(g3 < 0) | (g3 > 7.5)] = np.nan
    ax2.plot_surface(g1, g2, g3, color="0.4", alpha=0.25, lw=0)
    for clase, marcador, color in ((-1, "o", C_NEG), (1, "^", C_POS)):
        ax2.scatter(*F[y_circ == clase].T, marker=marcador, s=70, color=color, ec="white", depthshade=False)
    ax2.set(xlim=(0, 7.5), ylim=(-5.5, 5.5), zlim=(0, 7.5), xlabel="$x_1^2$", ylabel=r"$\sqrt{2}\,x_1 x_2$", zlabel="$x_2^2$",
            title="Espacio de ϕ: la frontera es un plano")
    ax2.view_init(elev=elevacion, azim=azimut)
    plt.tight_layout()
    plt.show()

w = [-0.561 -0.009 -0.572],  b = 1.777
Diferencia máxima entre w^T ϕ(x) + b y el score de SVC: 1.3e-15


In [28]:
interact(ver_levantar,
         elevacion=widgets.IntSlider(min=0, max=90, value=20, description="Elevación"),
         azimut=widgets.IntSlider(min=-180, max=180, value=-60, description="Azimut"));

interactive(children=(IntSlider(value=20, description='Elevación', max=90), IntSlider(value=-60, description='…

### El kernel gaussiano como medida de similitud (Figura 6)

$$
K(\mathbf{x}, \mathbf{z}) = \exp\left(-\frac{\|\mathbf{x} - \mathbf{z}\|^2}{2\sigma^2}\right)
$$

A la izquierda, $K$ en función de la distancia entre los puntos, con las tres curvas de la Figura 6 como referencia. A la derecha, la similitud de cada punto del plano con un punto fijo $\mathbf{z}$ (la estrella); el círculo punteado está a distancia $\sigma$ de $\mathbf{z}$, donde $K \approx 0.61$.

Qué observar: $\sigma$ es la distancia a la que los puntos dejan de parecerse. Con $\sigma$ pequeño, solo los puntos muy cercanos a $\mathbf{z}$ tienen similitud apreciable.

In [29]:
def gaussiano(X, Z, sigma):
    """Matriz con K(x, z) = exp(-||x - z||² / (2σ²)) para cada fila x de X y z de Z."""
    d2 = ((X[:, None, :] - Z[None, :, :]) ** 2).sum(axis=-1)
    return np.exp(-d2 / (2 * sigma ** 2))


def ver_similitud(sigma=1.0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1.2, 1]))
    r = np.linspace(0, 5, 400)
    for s, color in zip((0.5, 1, 2), (C_NEG, VERDE, C_POS)):
        ax1.plot(r, np.exp(-r ** 2 / (2 * s ** 2)), color=color, lw=1.6, alpha=0.45, label=f"σ = {s}")
    ax1.plot(r, np.exp(-r ** 2 / (2 * sigma ** 2)), color="black", lw=3, label=f"σ = {sigma:.2f}")
    ax1.axvline(sigma, color="black", ls=":", lw=1.5)
    ax1.set(xlim=(0, 5), ylim=(0, 1.03), xlabel=r"$\|\mathbf{x} - \mathbf{z}\|$", ylabel=r"$K(\mathbf{x}, \mathbf{z})$",
            title="Similitud en función de la distancia")
    ax1.legend(loc="upper right")

    z = np.array([[0.0, 0.0]])
    lim = (-3, 3)
    g1, g2 = np.meshgrid(np.linspace(*lim, 250), np.linspace(*lim, 250))
    K = gaussiano(np.column_stack([g1.ravel(), g2.ravel()]), z, sigma).reshape(g1.shape)
    im = ax2.imshow(K, extent=(*lim, *lim), origin="lower", cmap="Purples", vmin=0, vmax=1)
    ax2.add_patch(plt.Circle((0, 0), sigma, fill=False, ls=":", lw=2, color="black"))
    ax2.plot(0, 0, "*", color="gold", ms=20, mec="black")
    fig.colorbar(im, ax=ax2, fraction=0.046, label=r"$K(\mathbf{x}, \mathbf{z})$")
    ax2.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$", title=r"Similitud con $\mathbf{z}$")
    plt.tight_layout()
    plt.show()

In [30]:
interact(ver_similitud, sigma=widgets.FloatLogSlider(value=1, base=10, min=-1, max=0.5, step=0.02, description="σ"));

interactive(children=(FloatLogSlider(value=1.0, description='σ', max=0.5, min=-1.0, step=0.02), Output()), _do…

### La predicción como suma de campanas

Los datos de una sola feature de la Figura 5. Con el kernel gaussiano, el score de un punto nuevo es

$$
f(x) = \sum_{i=1}^m \alpha_i y^{(i)} K(x^{(i)}, x) + b,
$$

es decir, una campana centrada en cada vector de soporte, hacia arriba si es positivo y hacia abajo si es negativo, con altura $\alpha_i$. Las curvas delgadas son las campanas; la negra gruesa es su suma más $b$. Se predice $y = 1$ donde $f(x) > 0$ (fondo rojo). La SVM se entrena con $C = 10$, como en la Figura 7.

Qué observar:

- Los ejemplos que no son vectores de soporte no tienen campana: no aportan nada a la predicción.
- Con $\sigma$ grande las campanas son anchas y se mezclan: $f$ es suave y la frontera es simple, pero puede no separar bien los datos.
- Con $\sigma$ pequeño cada campana cubre solo a su ejemplo, todos se vuelven vectores de soporte, y lejos de los datos $f(x) \approx b$: el modelo solo "conoce" los puntos que vio. Es el overfitting de la Figura 7, en una dimensión.

In [31]:
# Los datos de la Figura 5
x_1d = np.array([-2.6, -2.1, -1.7, 1.6, 2.0, 2.5, -0.9, -0.5, -0.1, 0.3, 0.8])
y_1d = np.array([-1, -1, -1, -1, -1, -1, 1, 1, 1, 1, 1])


def ver_campanas(sigma=0.8):
    modelo = SVC(kernel="rbf", gamma=1 / (2 * sigma ** 2), C=10).fit(x_1d[:, None], y_1d)
    sv = modelo.support_vectors_.ravel()
    coef = modelo.dual_coef_.ravel()  # α_i y_i
    xs = np.linspace(-3.5, 3.5, 700)
    campanas = coef * np.exp(-(xs[:, None] - sv) ** 2 / (2 * sigma ** 2))
    f = campanas.sum(axis=1) + modelo.intercept_[0]
    tope = max(2.5, 1.15 * np.abs(campanas).max(), 1.15 * np.abs(f).max())

    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.fill_between(xs, -tope, tope, where=f >= 0, color=C_POS, alpha=0.12, lw=0)
    ax.fill_between(xs, -tope, tope, where=f < 0, color=C_NEG, alpha=0.1, lw=0)
    for j in range(len(sv)):
        ax.plot(xs, campanas[:, j], color=C_POS if coef[j] > 0 else C_NEG, lw=1.4, alpha=0.75)
    ax.plot(xs, f, color="black", lw=3, label=r"$f(x) = \sum_i \alpha_i y^{(i)} K(x^{(i)}, x) + b$")
    ax.axhline(0, color="0.4", lw=1)
    for c in (-1, 1):
        ax.axhline(c, color="0.4", ls="--", lw=1)
    puntos(ax, np.column_stack([x_1d, np.zeros_like(x_1d)]), y_1d, tam=150)
    soporte(ax, np.column_stack([sv, np.zeros_like(sv)]), tam=150)
    ax.set(xlim=(-3.5, 3.5), ylim=(-tope, tope), xlabel="$x$",
           title=f"σ = {sigma:.2f}:  {len(sv)} vectores de soporte de {len(y_1d)},  "
                 f"exactitud de entrenamiento = {modelo.score(x_1d[:, None], y_1d):.0%}")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
interact(ver_campanas, sigma=widgets.FloatLogSlider(value=0.8, base=10, min=-1.3, max=0.7, step=0.02, description="σ"));

interactive(children=(FloatLogSlider(value=0.8, description='σ', max=0.7, min=-1.3, step=0.02), Output()), _do…

### Máquinas de soporte vectorial con kernel gaussiano (Figura 7)

Los 120 ejemplos en forma de medialuna de la Figura 7. El fondo es el score $\sum_i \alpha_i y^{(i)} K(\mathbf{x}^{(i)}, \mathbf{x}) + b$, la curva negra es la frontera y las punteadas son $\text{score} = \pm 1$. A la derecha, la exactitud de entrenamiento y de prueba (sobre 1000 ejemplos nuevos de la misma distribución) para cada $\sigma$, con el $C$ escogido.

Qué observar:

- Con $\sigma$ grande la frontera es casi lineal y el modelo se queda corto (underfitting): las dos exactitudes son bajas.
- Al bajar $\sigma$ la frontera se vuelve más flexible. Con $\sigma$ muy pequeño rodea ejemplos individuales y casi todos son vectores de soporte: la exactitud de entrenamiento llega a 100%, pero la de prueba baja (overfitting).
- El mejor $\sigma$ está en el medio, y se escoge con validación. $C$ también cambia la complejidad: con $C$ grande, el modelo intenta clasificar bien todos los ejemplos de entrenamiento (Parte 5).

In [33]:
X_moon, y_moon = fig05.X_moon, fig05.y_moon
X_test, y_test = fig05.X_test, fig05.y_test
sigmas = [0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 5.0]
valores_C = [0.1, 1, 10, 100, 1000]


@lru_cache(maxsize=None)
def svm_gauss(sigma, C):
    return SVC(kernel="rbf", gamma=1 / (2 * sigma ** 2), C=C).fit(X_moon, y_moon)


def ver_gaussiano(sigma=0.5, C=10):
    modelo = svm_gauss(sigma, C)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw=dict(width_ratios=[1.4, 1]))
    fondo(ax1, modelo.decision_function, fig05.x_lim, fig05.y_lim)
    puntos(ax1, X_moon, y_moon, tam=50)
    soporte(ax1, modelo.support_vectors_, tam=50)
    ax1.set(xlabel="$x_1$", ylabel="$x_2$", title=f"σ = {sigma}, C = {C}:  {len(modelo.support_)} vectores de soporte de {len(y_moon)}")

    train = [svm_gauss(s, C).score(X_moon, y_moon) for s in sigmas]
    test = [svm_gauss(s, C).score(X_test, y_test) for s in sigmas]
    ax2.semilogx(sigmas, train, "o-", color="0.5", lw=2, label="entrenamiento")
    ax2.semilogx(sigmas, test, "s-", color=VERDE, lw=2.5, label="prueba")
    k = sigmas.index(sigma)
    ax2.axvline(sigma, color="black", ls=":", lw=1.5)
    ax2.set(ylim=(0.6, 1.02), xlabel="σ", ylabel="Exactitud",
            title=f"Entrenamiento: {porcentaje(train[k])},  prueba: {porcentaje(test[k])}")
    ax2.legend(loc="lower left")
    plt.tight_layout()
    plt.show()

In [34]:
interact(ver_gaussiano,
         sigma=widgets.SelectionSlider(options=sigmas, value=0.5, description="σ"),
         C=widgets.SelectionSlider(options=valores_C, value=10, description="C"));

interactive(children=(SelectionSlider(description='σ', index=4, options=(0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 5…

### Kernels válidos y el teorema de Mercer

Tomamos $n$ ejemplos al azar de las medialunas, formamos la matriz kernel $K_{ij} = K(\mathbf{x}^{(i)}, \mathbf{x}^{(j)})$ (izquierda) y calculamos sus autovalores (derecha). Una matriz simétrica es semidefinida positiva exactamente cuando todos sus autovalores son $\geq 0$.

Qué observar:

- Con los kernels válidos (gaussiano, polinomial, lineal) nunca aparece un autovalor negativo, sin importar cuántos puntos tomemos.
- Con el kernel lineal hay a lo sumo dos autovalores distintos de cero, porque $\phi(\mathbf{x}) = \mathbf{x}$ tiene solo dos componentes.
- Con $K(\mathbf{x}, \mathbf{z}) = \|\mathbf{x} - \mathbf{z}\|^2$ aparecen autovalores negativos (en rojo) desde $n = 2$, como en el ejemplo de las notas: no es un kernel válido. Si $\mathbf{v}$ es el autovector de un autovalor $\lambda < 0$, entonces $\mathbf{v}^T K \mathbf{v} = \lambda < 0$.

In [20]:
def distancia2(X, Z):
    return ((X[:, None, :] - Z[None, :, :]) ** 2).sum(axis=-1)


kernels = {
    "Gaussiano, σ = 0.5": lambda X, Z: gaussiano(X, Z, 0.5),
    "Polinomial, (xᵀz + 1)²": lambda X, Z: (X @ Z.T + 1) ** 2,
    "Lineal, xᵀz": lambda X, Z: X @ Z.T,
    "‖x − z‖²": distancia2,
}
orden_mercer = np.random.default_rng(1).permutation(len(X_moon))


def ver_mercer(kernel="Gaussiano, σ = 0.5", n=8):
    X = X_moon[orden_mercer[:n]]
    K = kernels[kernel](X, X)
    autovalores = np.linalg.eigvalsh(K)[::-1]
    minimo = autovalores.min()
    tol = 1e-9 * np.abs(autovalores).max()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1, 1.2]))
    im = ax1.imshow(K, cmap="Purples")
    fig.colorbar(im, ax=ax1, fraction=0.046)
    ax1.set(xticks=[], yticks=[], xlabel="$j$", ylabel="$i$", title=f"Matriz kernel {n} × {n}")

    colores = np.where(autovalores < -tol, C_POS, "0.45")
    ax2.bar(np.arange(1, n + 1), autovalores, color=colores)
    ax2.axhline(0, color="black", lw=1)
    ax2.set(xlabel="Autovalor (de mayor a menor)", ylabel="Valor", xticks=range(1, n + 1, max(1, n // 10)))
    if minimo < -tol:
        ax2.set_title(f"Mínimo autovalor = {minimo:.3f} < 0:\nno es semidefinida positiva", color=C_POS)
    else:
        ax2.set_title(f"Mínimo autovalor = {max(minimo, 0):.3f}:\nsemidefinida positiva")
    plt.tight_layout()
    plt.show()

In [21]:
interact(ver_mercer,
         kernel=widgets.ToggleButtons(options=list(kernels), description="Kernel"),
         n=widgets.IntSlider(min=2, max=40, value=8, description="n"));

interactive(children=(ToggleButtons(description='Kernel', options=('Gaussiano, σ = 0.5', 'Polinomial, (xᵀz + 1…

## 5. Margen suave

### El problema de los outliers (Figura 8)

Los datos de la Figura 8 con el ejemplo negativo atípico, que se puede mover con los sliders. La recta gris clara es la frontera de margen óptimo sin el outlier (panel izquierdo de la Figura 8). El outlier lleva un anillo verde. A la derecha, el ancho del margen $2/\|\mathbf{w}\|$ en función de $C$, para la posición actual del outlier.

Qué observar:

- Con $C = 10^6$ (margen duro, como el panel central de la Figura 8), la frontera gira para dejar el outlier del lado correcto y la calle se angosta.
- Al bajar $C$, la SVM acepta que el outlier quede dentro del margen o mal clasificado, y recupera una calle ancha parecida a la original (panel derecho de la Figura 8, con $C = 0.5$).
- Si llevamos el outlier dentro de los positivos, los datos dejan de ser separables: el margen duro ya no existe, pero el margen suave sigue funcionando.

In [22]:
X_base, y_base = fig05.X_base, fig05.y_base
modelo_base = svm_lineal(X_base, y_base)
valores_C_out = [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 100, 1000, C_DURO]
etiquetas_C_out = [(f"{C:g}" if C < C_DURO else "10⁶ (duro)", C) for C in valores_C_out]


@lru_cache(maxsize=None)
def svm_outlier(ox, oy, C):
    X = np.vstack([X_base, [ox, oy]])
    y = np.append(y_base, -1)
    return svm_lineal(X, y, C), X, y


def ver_outlier(ox=2.35, oy=3.4, C=C_DURO):
    modelo, X, y = svm_outlier(ox, oy, C)
    w, b = modelo.coef_.ravel(), modelo.intercept_[0]
    errores = int(np.sum(y * (X @ w + b) < 0))
    no_separable = C == C_DURO and np.any(y * (X @ w + b) < 1 - 1e-3)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    recta(ax1, modelo_base.coef_.ravel(), modelo_base.intercept_[0], color="0.75", lw=2)
    fondo(ax1, lambda P: P @ w + b, (0, 5), (0, 5), lineas=False)
    calle(ax1, w, b)
    puntos(ax1, X, y)
    soporte(ax1, modelo.support_vectors_)
    ax1.scatter(ox, oy, s=500, facecolor="none", ec=VERDE, lw=2.5, zorder=5)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$",
            title=f"Ancho = {2 / np.linalg.norm(w):.2f},  {len(modelo.support_)} vectores de soporte,  {errores} mal clasificados")
    if no_separable:
        ax1.text(0.5, 0.02, "No es separable: el margen duro no existe", transform=ax1.transAxes, ha="center",
                 fontsize=12, color=C_POS, bbox=dict(fc="white", ec=C_POS))

    anchos = [2 / np.linalg.norm(svm_outlier(ox, oy, c)[0].coef_) for c in valores_C_out]
    ax2.semilogx(valores_C_out, anchos, "o-", color=NARANJA, lw=2.5)
    ax2.semilogx(C, 2 / np.linalg.norm(w), "o", color="black", ms=11)
    ax2.axhline(2 / np.linalg.norm(modelo_base.coef_), color="0.6", ls=":", lw=2, label="margen duro sin outlier")
    ax2.set(xlabel="C", ylabel=r"Ancho del margen $2/\|\mathbf{w}\|$", ylim=(0, None), title="Ancho del margen según C")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [23]:
interact(ver_outlier,
         ox=widgets.FloatSlider(min=0.5, max=4.5, step=0.05, value=2.35, description="Outlier x₁"),
         oy=widgets.FloatSlider(min=0.5, max=4.5, step=0.05, value=3.4, description="Outlier x₂"),
         C=widgets.SelectionSlider(options=etiquetas_C_out, value=C_DURO, description="C"));

interactive(children=(FloatSlider(value=2.35, description='Outlier x₁', max=4.5, min=0.5, step=0.05), FloatSli…

### Efecto de $C$ y variables de holgura (Figura 9)

Los 60 ejemplos de la Figura 9, cuyas clases se solapan. Para cada ejemplo con holgura $\xi_i = \max(0,\ 1 - y^{(i)}(\mathbf{w}^T\mathbf{x}^{(i)} + b)) > 0$, el segmento naranja va del ejemplo hasta la recta punteada de su clase; su largo es $\xi_i / \|\mathbf{w}\|$. Los vectores de soporte se marcan según las condiciones de complementariedad:

- círculo negro: $0 < \alpha_i < C$, exactamente sobre el margen;
- círculo naranja: $\alpha_i = C$, dentro del margen o mal clasificado.

Qué observar al mover $C$:

- Con $C$ pequeño la calle es ancha y muchos ejemplos quedan dentro de ella: casi todos son vectores de soporte, la mayoría con $\alpha_i = C$.
- Con $C$ grande cada violación cuesta mucho: la calle se angosta, las holguras se achican y quedan pocos vectores de soporte.
- A la derecha, el ancho del margen y el número de vectores de soporte en función de $C$. Dividiendo el objetivo entre $C$, $1/C$ hace el papel de $\lambda$: moverse a la izquierda es regularizar más.

In [ ]:
X_c, y_c = fig05.X_c, fig05.y_c
m_c = len(y_c)
lim_c = (-0.5, 5.5)
exponentes_C = np.round(np.arange(-2, 2.01, 0.125), 3)


@lru_cache(maxsize=None)
def svm_c(exponente):
    C = 10.0 ** exponente
    modelo = svm_lineal(X_c, y_c, C)
    a = alphas(modelo, m_c)
    acotados = np.isclose(a, C, rtol=1e-3)
    return modelo, a, acotados


curva_ancho = [2 / np.linalg.norm(svm_c(e)[0].coef_) for e in exponentes_C]
curva_sv = [len(svm_c(e)[0].support_) for e in exponentes_C]
curva_acotados = [svm_c(e)[2].sum() for e in exponentes_C]


def ver_efecto_C(exponente=-1.0):
    exponente = float(np.round(exponente, 3))
    C = 10.0 ** exponente
    modelo, a, acotados = svm_c(exponente)
    w, b = modelo.coef_.ravel(), modelo.intercept_[0]
    xi = np.maximum(0, 1 - y_c * (X_c @ w + b))

    fig = plt.figure(figsize=(13, 5.6))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.3, 1])
    ax1 = fig.add_subplot(grid[:, 0])
    fondo(ax1, lambda P: P @ w + b, lim_c, lim_c, lineas=False)
    calle(ax1, w, b, lim_c, lim_c)
    # Holgura: del ejemplo a la recta w^T x + b = y, en la dirección de w
    destino = X_c + (y_c * xi)[:, None] * w / (w @ w)
    for i in np.flatnonzero(xi > 1e-6):
        ax1.plot(*np.column_stack([X_c[i], destino[i]]), color=NARANJA, lw=2, zorder=2)
    puntos(ax1, X_c, y_c, tam=70)
    soporte(ax1, X_c[(a > 1e-8) & ~acotados], tam=70)
    soporte(ax1, X_c[acotados], tam=70, color=NARANJA)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$",
            title=f"C = {C:.3g}:  " + rf"$\frac{{1}}{{2}}\|\mathbf{{w}}\|^2$ = {0.5 * w @ w:.2f},  $C\sum\xi_i$ = {C * xi.sum():.2f}")

    ax2 = fig.add_subplot(grid[0, 1])
    ax2.semilogx(10.0 ** exponentes_C, curva_ancho, color=C_NEG, lw=2.5)
    ax2.semilogx(C, 2 / np.linalg.norm(w), "o", color=C_NEG, ms=10)
    ax2.set(ylabel=r"$2/\|\mathbf{w}\|$", title=f"Ancho del margen: {2 / np.linalg.norm(w):.2f}", ylim=(0, None))
    ax3 = fig.add_subplot(grid[1, 1], sharex=ax2)
    ax3.semilogx(10.0 ** exponentes_C, curva_sv, color="black", lw=2.5, label="todos")
    ax3.semilogx(10.0 ** exponentes_C, curva_acotados, color=NARANJA, lw=2.5, label=r"con $\alpha_i = C$")
    ax3.semilogx([C, C], [len(modelo.support_), acotados.sum()], "o", color="black", ms=9)
    ax3.set(xlabel="C", ylabel="Vectores de soporte", ylim=(0, m_c), title=f"{len(modelo.support_)} vectores de soporte ({acotados.sum()} con α = C)")
    ax3.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
interact(ver_efecto_C, exponente=widgets.FloatSlider(min=-2, max=2, step=0.125, value=-1, description="log₁₀ C"));